# ACHTUNG: In diesem Notebook werden händisch Werte nachgetragen. Am besten Notebook nachvollziehen und mit dem nächsten Datensatz im nächsten Notebook weitermachen!

### Prüfen NAs und 0 Values der Spalte 'rider_points_season' 

In [1]:
import pandas as pd
import os


In [ ]:
df = pd.read_pickle('../../data/processed/14_cleaned_master_data.pkl')

In [3]:
df.isna().sum()

race                                0
year                                0
url                                 0
rank                                0
rider_url                           0
time_gap                            0
birthdate                           0
height                              0
name                                0
nationality                         0
weight                              0
url_name                            0
departure                           0
arrival                             0
distance                            0
vertical_meters                     0
won_how                             0
avg_speed                           0
race_ranking                        0
one_day_races                       0
gc                                  0
time_trial                          0
sprint                              0
climber                             0
hills                               0
stage_nr                            0
date        

In [ ]:
# Nullen und NaNs für die Saisonpunkte zählen
points_nans = df['rider_points_season'].isna().sum()
points_zeros = (df['rider_points_season'] == 0).sum()

print(f"In der Spalte 'rider_points_season' gibt es:")
print(f"  - Fehlende Werte (NaN): {points_nans} Zeilen")
print(f"  - Null-Werte (0):       {points_zeros} Zeilen\n")

# Betroffene Zeilen isolieren (entweder NaN oder echte 0)
missing_points_df = df[(df['rider_points_season'].isna()) | (df['rider_points_season'] == 0)]

# Anzahl der einzigartigen Fahrer ermitteln (über rider_url)
unique_affected_riders = missing_points_df['rider_url'].nunique()
print(f"Davon betroffen sind: {unique_affected_riders} einzigartige Fahrer.")


In der Spalte 'rider_points_season' gibt es:
  - Fehlende Werte (NaN): 1643 Zeilen
  - Null-Werte (0):       0 Zeilen

Davon betroffen sind: 36 einzigartige Fahrer.


Ausgabe der 36 fehlenden Fahrer

In [8]:
missing_points = df[df['rider_points_season'].isna()]

recherche_points_df = missing_points[['name', 'year', 'nationality', 'rider_url']].drop_duplicates().copy()


recherche_points_df['rider_points_season_recherchiert'] = ""


recherche_points_df = recherche_points_df.sort_values(by=['name', 'year'])

target_dir = "../../data/interim"
os.makedirs(target_dir, exist_ok=True)

target_path = os.path.join(target_dir, "36_fahrer_points_recherche.csv")
recherche_points_df.to_csv(target_path, index=False, sep=';')

Händisch nachgetragen aus procyclingstats


In [11]:
recherche_points = pd.read_csv('../../data/interim/36_fahrer_points_recherche.csv', sep=';')


recherche_points['rider_points_season_recherchiert'] = pd.to_numeric(
    recherche_points['rider_points_season_recherchiert'], errors='coerce'
)


df = df.merge(
    recherche_points[['rider_url', 'year', 'rider_points_season_recherchiert']],
    on=['rider_url', 'year'],
    how='left'
)


df['rider_points_season'] = df['rider_points_season'].combine_first(df['rider_points_season_recherchiert'])


df.drop(columns=['rider_points_season_recherchiert'], inplace=True)




noch_nan_rows = df['rider_points_season'].isna().sum()


rest_missing = df[df['rider_points_season'].isna()]
noch_nan_kombis = rest_missing[['rider_url', 'year']].drop_duplicates().shape[0]

print(f"Verbleibende Zeilen im DF mit NaN-Punkten: {noch_nan_rows}")
print(f"Verbleibende offene Fahrer-Jahr-Kombis:    {noch_nan_kombis}")

Verbleibende Zeilen im DF mit NaN-Punkten: 0
Verbleibende offene Fahrer-Jahr-Kombis:    0


Wir tragen für die fehlenden Werte 0 ein
Begründung:
- sind mitgefahren 
- Ergebnisse waren nicht gut genug für Ranglistenpunkte

In [12]:
df.isna().sum()

race                                0
year                                0
url                                 0
rank                                0
rider_url                           0
time_gap                            0
birthdate                           0
height                              0
name                                0
nationality                         0
weight                              0
url_name                            0
departure                           0
arrival                             0
distance                            0
vertical_meters                     0
won_how                             0
avg_speed                           0
race_ranking                        0
one_day_races                       0
gc                                  0
time_trial                          0
sprint                              0
climber                             0
hills                               0
stage_nr                            0
date        

### Checkpoint
Neues .pkl erstellen 15

In [13]:
pfad = '../../data/processed/15_cleaned_master_data.pkl'
df.to_pickle(pfad)